In [1]:
import random
import time
import gc
import joblib
import implicit
import requests
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

try:
    import spacy
    spacy.load("en_core_web_sm")
except Exception:
    import subprocess, sys, importlib
    print('en_core_web_sm not found — installing into current environment...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'en-core-web-sm==3.8.0'])
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    importlib.invalidate_caches()
    importlib.reload(__import__('spacy'))
    print('Installation complete — you can now run: nlp = spacy.load("en_core_web_sm")')

try:
    from sklearn.ensemble import RandomForestClassifier as RFC
except:
    !pip install --quiet scikit-learn
    from sklearn.ensemble import RandomForestClassifier as RFC

import scipy.sparse as sp
from statsmodels.stats.proportion import test_proportions_2indep, samplesize_proportions_2indep_onetail
from pandas.api.types import CategoricalDtype

/home/yaroslav/anaconda3/envs/notebooks/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Data Preparation

In [3]:
books = pd.read_csv('./data/books.csv')
ratings = pd.read_csv('./data/ratings.csv')
ratings.head(5)

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [4]:
def preprpcess_data(
    ratings: pd.DataFrame,
    train_ratio: float = 0.7

):
    dataset = ratings.copy()

    dataset['implicit_rating'] = dataset.rating \
        .apply(
            lambda x: 1 if x >= 4 else 0
        )

    dataset['book_order'] = dataset \
        .sort_values(['user_id']) \
        .groupby(['user_id']) \
        .cumcount() + 1

    dataset['total_books'] = (
        dataset.groupby('user_id')['book_id'] \
        .transform('count')
    )

    dataset['fraction'] = (
        dataset['book_order']
        / dataset['total_books']
    )

    train = dataset[
        dataset['fraction'] <= train_ratio
    ].copy()

    train['confidence'] = np.log1p(
        train['implicit_rating'] * 10
    )

    test = dataset[
        dataset['fraction'] > train_ratio
    ].copy()

    test['confidence'] = np.log1p(
        test['implicit_rating'] * 10
    )
    
    del dataset
    gc.collect()

    return train, test

In [5]:
train, test = preprpcess_data(ratings)
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4159553 entries, 0 to 5976478
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   user_id          int64  
 1   book_id          int64  
 2   rating           int64  
 3   implicit_rating  int64  
 4   book_order       int64  
 5   total_books      int64  
 6   fraction         float64
 7   confidence       float64
dtypes: float64(2), int64(6)
memory usage: 285.6 MB


### Feature enginering

In [5]:
books.head(5)

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,language_code,average_rating,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",eng,4.34,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",en-US,3.57,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,eng,4.25,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,eng,3.89,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [6]:
def simple_ohe(
        genre_dict,
        genre
):
    if genre in genre_dict:
        return 1
    else:
        return 0

In [7]:
genres_df = pd.read_json('./data/goodreads_book_genres_initial.json', lines=True)
genres_df.head(5)

,book_id,genres
0,5333265,"{'history, historical fiction, biography': 1}"
1,1333909,"{'fiction': 219, 'history, historical fiction,..."
2,7327624,"{'fantasy, paranormal': 31, 'fiction': 8, 'mys..."
3,6066819,"{'fiction': 555, 'romance': 23, 'mystery, thri..."
4,287140,{'non-fiction': 3}


In [8]:
genres_df = genres_df[genres_df.book_id.isin(books.goodreads_book_id)]
genres_df.columns = ['book_id', 'genre_dict']
genres_df.head(5)

,book_id,genre_dict
3,6066819,"{'fiction': 555, 'romance': 23, 'mystery, thri..."
15,89375,"{'non-fiction': 534, 'history, historical fict..."
583,54270,"{'history, historical fiction, biography': 108..."
807,38568,"{'fantasy, paranormal': 1907, 'romance': 1598,..."
816,38562,"{'fantasy, paranormal': 1002, 'romance': 896, ..."


In [9]:
all_genres = set()
for dict_genre in genres_df.genre_dict:
    for elem in list(dict_genre.keys()):
        all_genres.add(elem)

print(f'genres total: {len(all_genres)}')
for genre in all_genres:
    genres_df[genre] = 0

for genre in all_genres:
    genres_df[genre] = genres_df.apply(
        lambda df: simple_ohe(df['genre_dict'], genre), axis=1
    )


books_profile = books[['book_id', 'goodreads_book_id']]
train = train.merge(
    books_profile,
    left_on='book_id',
    right_on='book_id',
    how='left'
)

books_profile = books_profile.merge(
    genres_df,
    left_on='goodreads_book_id',
    right_on='book_id',
    how='left'
)

train = train.merge(
    genres_df,
    left_on='goodreads_book_id',
    right_on='book_id',
    how='left'
)

users_profiles = train.groupby('user_id')[list(all_genres)].sum()

train.head(5)
print(books.shape)

genres total: 10
(10000, 23)


In [10]:
users_profiles.columns = [
    'user_' + name for name in list(users_profiles)
]
train.columns = [
    'book_' + item if item in all_genres else item for item in list(train)
]
books_profile.columns = [
    'book_' + item if item in all_genres else item for item in list(books_profile)
]

books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   book_id                    10000 non-null  int64  
 1   goodreads_book_id          10000 non-null  int64  
 2   best_book_id               10000 non-null  int64  
 3   work_id                    10000 non-null  int64  
 4   books_count                10000 non-null  int64  
 5   isbn                       9300 non-null   object 
 6   isbn13                     9415 non-null   float64
 7   authors                    10000 non-null  object 
 8   original_publication_year  9979 non-null   float64
 9   original_title             9415 non-null   object 
 10  title                      10000 non-null  object 
 11  language_code              8916 non-null   object 
 12  average_rating             10000 non-null  float64
 13  ratings_count              10000 non-null  int6

In [11]:
nlp = spacy.load("en_core_web_sm") # load nlp tokenizer and vectorizer
books['vector_representation'] = \
    books.title.apply(
        lambda x: nlp(x).vector
    )

new_vectors_columns_name = [
    f'vec_component_{num}' for num in range(96)
]

books[new_vectors_columns_name] = pd.DataFrame(
    books.vector_representation.tolist(), index=books.index
)

train = train.merge(
    books[new_vectors_columns_name + ['book_id']],
    left_on='book_id_x',
    right_on = 'book_id'
)

books_profile = books_profile.merge(
    books[new_vectors_columns_name + ['book_id']],
    left_on='book_id_x',
    right_on = 'book_id'
)

train = train.merge(
    users_profiles,
    left_on='user_id',
    right_on='user_id',
    how='left'
)

# train.head(5)

In [12]:
cols_for_using = [
 'book_mystery, thriller, crime',
 'book_non-fiction',
 'book_romance',
 'book_fantasy, paranormal',
 'book_poetry',
 'book_fiction',
 'book_young-adult',
 'book_history, historical fiction, biography',
 'book_comics, graphic',
 'book_children',
 'user_mystery, thriller, crime',
 'user_non-fiction',
 'user_romance',
 'user_fantasy, paranormal',
 'user_poetry',
 'user_fiction',
 'user_young-adult',
 'user_history, historical fiction, biography',
 'user_comics, graphic',
 'user_children']+new_vectors_columns_name

train.fillna(0, inplace=True)
classifier = RFC(n_estimators=20, criterion = 'entropy', random_state=42)

classifier.fit(train[cols_for_using], train['implicit_rating'])

,n_estimators,20
,criterion,'entropy'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
joblib.dump(books_profile, './joblib/books_profile.pkl') # save books profile for content-based filtering
joblib.dump(users_profiles, './joblib/users_profiles.pkl') # save users profiles for content-based filtering
joblib.dump(cols_for_using, './joblib/cols_for_using.joblib') # save columns names for using in content-based filtering
joblib.dump(classifier, './joblib/rf_classifier.joblib') # save classifier for using in content-based filtering (predicting implicit rating)

del genres_df, classifier, books_profile, users_profiles
gc.collect()

292

In [14]:
user_index = train.user_id.unique()
books_index = train.book_id.unique()

rows = train['user_id'].astype(CategoricalDtype(categories=user_index)).cat.codes
cols = train['book_id'].astype(CategoricalDtype(categories=books_index)).cat.codes

matrix = sp.csr_matrix(
    (train.confidence, (rows, cols)),
    shape=(len(user_index), len(books_index))
)

matrix = matrix.toarray()

for ind, score in enumerate(matrix[50]):
    if score >= 1:
        book_id = books_index[ind]
        print(score, books[books['book_id'] == book_id].original_title.values[0])

most_popular_book = list(ratings.book_id.value_counts().reset_index()[:40].book_id)

read_history = train[['user_id', 'book_id_x']] \
    .groupby('user_id').agg({'book_id_x': list})

del train, books, ratings
gc.collect()

read_history = read_history.reset_index()

read_history_dict = dict(zip(
    read_history.user_id, read_history.book_id_x
))


als_model = implicit.als.AlternatingLeastSquares(
    factors=128,
    iterations = 120
)

als_model.fit(sp.csr_matrix(matrix))

2.3978952727983707 Harry Potter and the Prisoner of Azkaban
2.3978952727983707 Harry Potter and the Half-Blood Prince
2.3978952727983707 Harry Potter and the Order of the Phoenix
2.3978952727983707 Harry Potter and the Chamber of Secrets
2.3978952727983707 Harry Potter and the Goblet of Fire
2.3978952727983707 Life of Pi
2.3978952727983707 The History of Love
2.3978952727983707 Un di Velt Hot Geshvign
2.3978952727983707 The Kite Runner 
2.3978952727983707 Me Talk Pretty One Day
2.3978952727983707 Sense and Sensibility
2.3978952727983707 Gone with the Wind
2.3978952727983707 The Audacity of Hope: Thoughts on Reclaiming the American Dream
2.3978952727983707 ノルウェイの森 [Noruwei no Mori]
2.3978952727983707 Freakonomics: A Rogue Economist Explores the Hidden Side of Everything
2.3978952727983707 The Tipping Point: How Little Things Can Make a Big Difference
2.3978952727983707 Skinny Legs and All
2.3978952727983707 Dreams from My Father
2.3978952727983707 The Secret Life of Bees
2.3978952727983

/home/yaroslav/anaconda3/envs/notebooks/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/120 [00:00<?, ?it/s]

In [15]:
joblib.dump(matrix, './joblib/user_item_matrix.joblib') # save user-book matrix for collaborative filtering (ALS)
joblib.dump(books_index, './joblib/books_index.joblib') # save books index for using in collaborative filtering (ALS)
joblib.dump(user_index, './joblib/user_index.joblib') # save users index for using in collaborative filtering (ALS)

joblib.dump(most_popular_book, './joblib/most_popular_book.joblib') # save 40 most popular books for using in case of cold start (when we don't have any information about user)
joblib.dump(read_history_dict, './joblib/read_history_dict.joblib') # save read history for each user for using in case of cold start (when we don't have any information about user, but we know what books he read)
joblib.dump(als_model, './joblib/als_model.joblib') # save ALS model for using in collaborative filtering (predicting implicit rating for user-book pair)

['./joblib/als_model.joblib']

### AB test

In [6]:
%%time

already_checked = set()
conversion_list = []
while len(already_checked) < 1000:
    user_id = random.randint(1, 5342)
    if user_id in already_checked:
        continue
    already_checked.add(user_id)
    resp = requests.post(
        'http://127.0.0.1:5509/recommender',
        json={'model': 'most_popular_model', 'user_id': user_id}
    )
    # safe parse: default to empty list if server returns non-JSON or missing key
    try:
        recommendations = resp.json().get('result', [])
    except Exception:
        recommendations = []
    true_books = list(test[
        (test.user_id==user_id)&(test.implicit_rating==1)
    ].book_id.values)
    # skip users without true books or when no recommendations returned
    if len(true_books) == 0 or len(recommendations) == 0:
        continue
    conversion_for_user = len([book for book in recommendations if book in true_books]) / len(true_books)
    conversion_list.append(conversion_for_user)

print(np.mean(conversion_list))

0.10375197952157678
CPU times: user 25.5 s, sys: 1.48 s, total: 27 s
Wall time: 26.6 s


In [ ]:
# Текущая конверсия 2.0%, предполагаем что модель улучшит ее до 3.0%,
#  и подсчитаем сколько необходимо собрать пользователей для подсчета значимого результата
# мощность - 80%, уровень значимости  0.95
# https://www.evanmiller.org/ab-testing/sample-size.html дает размер выборки в 3.5 пользователей

In [12]:
# проведем AA тест 
%time 

conversion_list_group_a = []
stats_for_group_a = []
conversion_list_group_a_2 = []
stats_for_group_a2 = []

while len(conversion_list_group_a)<3500: 
    user_id = random.randint(1, 53424)
    if user_id in already_checked: 
        continue
    else:
        already_checked.add(user_id)
        group_for_user = random.randint(0,1)
        if group_for_user == 0: 
            recommendatations = requests.post('http://127.0.0.1:5509/recommender', json={"model": "most_popular_model", "user_id":user_id})
            recommendatations = recommendatations.json()['result']
            true_books = list(test[(test.user_id==user_id)&(test.implicit_rating==1)].book_id.values)
            conversion_for_user = len([book for book in recommendatations if book in true_books])/len(recommendatations)
            stats_for_user = [1 if book in true_books else 0 for book in recommendatations]
            stats_for_group_a.extend(stats_for_user)
            conversion_list_group_a.append(conversion_for_user)    
        else: 
            recommendatations = requests.post('http://127.0.0.1:5509/recommender', json={"model": "most_popular_model", "user_id":user_id})
            recommendatations = recommendatations.json()['result']
            true_books = list(test[(test.user_id==user_id)&(test.implicit_rating==1)].book_id.values)
            conversion_for_user = len([book for book in recommendatations if book in true_books])/len(recommendatations)
            stats_for_user = [1 if book in true_books else 0 for book in recommendatations]
            stats_for_group_a2.extend(stats_for_user)
            conversion_list_group_a_2.append(conversion_for_user)    
            

print(np.mean(conversion_list_group_a), np.mean(conversion_list_group_a_2))

CPU times: user 10 μs, sys: 2 μs, total: 12 μs
Wall time: 25 μs


0.07906857208510785 0.07875003513423866


In [13]:
# конверсия будто бы не отличаетсz, но применим стат критерий
# from statsmodels.stats.proportion import test_proportions_2indep

result = test_proportions_2indep(count1=sum(stats_for_group_a), nobs1=len(stats_for_group_a), 
                                 count2=sum(stats_for_group_a2), nobs2=len(stats_for_group_a2))
result

<class 'statsmodels.stats.base.HolderTuple'>
statistic = np.float64(0.08355134010090204)
pvalue = np.float64(0.933413156562374)
compare = 'diff'
method = 'agresti-caffo'
diff = np.float64(8.97398562349494e-05)
ratio = np.float64(1.0012390025879987)
odds_ratio = np.float64(1.0013358789981037)
variance = np.float64(1.1581424524225502e-06)
alternative = 'two-sided'
value = 0
tuple = (np.float64(0.08355134010090204), np.float64(0.933413156562374))

In [ ]:
# Видим что p-va.ue > 0.05 т.е не можем отклонить гипотезу о равенстве средних, в группах нет различий, теперь попробуем провести АВ тест 

In [ ]:
# проведем AВ тест 
%time

conversion_list_group_a = []
stats_for_group_a = []
conversion_list_group_b = []
stats_for_group_b = []
user_index = joblib.load('./joblib/user_index.joblib')

while len(conversion_list_group_a)<3500: 
    user_id = random.randint(1, 53424)
    if user_id in already_checked: 
        continue
    else:
        already_checked.add(user_id)
        group_for_user = random.randint(0,1)
        if group_for_user == 0: 
            recommendatations = requests.post('http://127.0.0.1:5509/recommender', json={"model": "most_popular_model", "user_id":user_id})
            recommendatations = recommendatations.json()['result']
            if len(recommendatations) == 0: 
                continue 
            else:
                true_books = list(test[(test.user_id==user_id)&(test.implicit_rating==1)].book_id.values)
                conversion_for_user = len([book for book in recommendatations if book in true_books])/len(recommendatations)
                stats_for_user = [1 if book in true_books else 0 for book in recommendatations]
                stats_for_group_a.extend(stats_for_user)
                conversion_list_group_a.append(conversion_for_user)    
        else: 
            user_ind = int(np.where(user_index==user_id)[0][0])
            recommendatations = requests.post('http://127.0.0.1:5509/recommender', json={"model": "hybrid_model", 
                                                                                         "user_id":user_id, "usr_ind":user_ind})
            recommendatations = recommendatations.json()['result']
            if len(recommendatations) == 0: 
                continue 
            else: 
                true_books = list(test[(test.user_id==user_id)&(test.implicit_rating==1)].book_id.values)
                conversion_for_user = len([book for book in recommendatations if book in true_books])/len(recommendatations)
                stats_for_user = [1 if book in true_books else 0 for book in recommendatations]
                stats_for_group_b.extend(stats_for_user)
                
                conversion_list_group_b.append(conversion_for_user)


print(np.mean(conversion_list_group_a), np.mean(conversion_list_group_b))

CPU times: user 7 μs, sys: 1 μs, total: 8 μs
Wall time: 16.7 μs
0.0794044811423364 0.17505155927551064


In [8]:
result = test_proportions_2indep(count1=sum(stats_for_group_a), nobs1=len(stats_for_group_a), 
                                 count2=sum(stats_for_group_b), nobs2=len(stats_for_group_b))
result


<class 'statsmodels.stats.base.HolderTuple'>
statistic = np.float64(-71.04629408424233)
pvalue = np.float64(0.0)
compare = 'diff'
method = 'agresti-caffo'
diff = np.float64(-0.10221746674437011)
ratio = np.float64(0.4154646785200125)
odds_ratio = np.float64(0.36966983800805836)
variance = np.float64(2.069953809070436e-06)
alternative = 'two-sided'
value = 0
tuple = (np.float64(-71.04629408424233), np.float64(0.0))